# CMGAN-George Training Notebook

Dieses Notebook startet das Training von `cmgan-george` direkt aus Jupyter.

Vor dem Start bitte sicherstellen:
- `__config__.py` zeigt auf deine lokalen Datenpfade.
- `models/huBERT_wrapper.py` verweist auf das richtige HuBERT-Modell.
- Die gewünschte Hyperparameter-Datei liegt unter `hparams/`.

## Standard-Setup

Die Zellen unten prüfen zuerst den Projektpfad und starten danach `train.py` mit einer Standard-Konfiguration.

In [ ]:
!python -m pip install "pip<24.1" "setuptools<70" wheel "cython<3"
!python -m pip install speechbrain==1.0.2 hyperpyyaml pesq tensorboard onnxruntime torchinfo pyloudnorm glob2 pyroomacoustics einops transformers torchaudio librosa
!python -m pip install git+https://github.com/schmiph2/pysepm.git
!python -m pip install fairseq

In [ ]:
from pathlib import Path
import os
import sys

candidate_roots = [
    Path.cwd(),
    Path.cwd() / 'cmgan-george',
    Path('/workspaces/MetricGAN-KAN-LwF/cmgan-george'),
]
repo_root = next((path for path in candidate_roots if (path / 'train.py').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Konnte cmgan-george/train.py nicht finden. Bitte das Notebook im Workspace öffnen.')

os.chdir(repo_root)
print(f'Repo root: {repo_root}')
print(f'Python: {sys.executable}')

hparams_file = Path('hparams/hyperparams_chime_bak_ovr_pesq_1.0.yaml')
if not hparams_file.exists():
    raise FileNotFoundError(f'Hyperparameter-Datei fehlt: {hparams_file}')

print(f'Using hparams: {hparams_file}')

# 1) Bevorzugt direkten Libri3Mix-Root finden (dein Setup: .../LibriMix/storage_dir/Libri3Mix).
explicit_libri3mix_root = os.environ.get('LIBRI3MIX_ROOT_PATH', '').strip()
candidate_libri3mix_roots = [
    Path(explicit_libri3mix_root) if explicit_libri3mix_root else None,
    repo_root / 'data' / 'libri' / 'LibriMix' / 'storage_dir' / 'Libri3Mix',
    repo_root.parent / 'data' / 'libri' / 'LibriMix' / 'storage_dir' / 'Libri3Mix',
    Path.home() / 'speechbrain' / 'data' / 'libri' / 'LibriMix' / 'storage_dir' / 'Libri3Mix',
    Path('/home/lugra002/speechbrain/data/libri/LibriMix/storage_dir/Libri3Mix'),
]

chosen_libri3mix_root = None
for root in candidate_libri3mix_roots:
    if root is None:
        continue
    train360 = root / 'wav16k' / 'max' / 'train-360'
    if train360.exists():
        chosen_libri3mix_root = root.resolve()
        break

if chosen_libri3mix_root is not None:
    os.environ['LIBRI3MIX_ROOT_PATH'] = chosen_libri3mix_root.as_posix()
    print('LIBRI3MIX_ROOT_PATH =', os.environ['LIBRI3MIX_ROOT_PATH'])
else:
    # 2) Fallback: CHIME2023_DATA_ROOT-Layout pruefen (.../data/LibriMix/Libri3Mix).
    explicit_data_root = os.environ.get('CHIME2023_DATA_ROOT', '').strip()
    candidate_data_roots = [
        Path(explicit_data_root) if explicit_data_root else None,
        repo_root / 'data',
        repo_root.parent / 'data',
        Path('/home/lugra002/speechbrain/lwf-multi/data'),
        Path('/mnt/parscratch/users/acp20glc/CHiME2023/data'),
    ]

    chosen_data_root = None
    for root in candidate_data_roots:
        if root is None:
            continue
        train360 = root / 'LibriMix' / 'Libri3Mix' / 'wav16k' / 'max' / 'train-360'
        if train360.exists():
            chosen_data_root = root.resolve()
            break

    if chosen_data_root is None:
        checked_libri = [str(p) for p in candidate_libri3mix_roots if p is not None]
        checked_data = [str(p) for p in candidate_data_roots if p is not None]
        raise FileNotFoundError(
            'Kein gueltiger Libri3Mix-Trainpfad gefunden. Gepruefte LIBRI3MIX_ROOT_PATH-Kandidaten: '
            + ', '.join(checked_libri)
            + '. Gepruefte CHIME2023_DATA_ROOT-Kandidaten: '
            + ', '.join(checked_data)
            + '. Setze LIBRI3MIX_ROOT_PATH direkt auf .../LibriMix/storage_dir/Libri3Mix.'
        )

    os.environ['CHIME2023_DATA_ROOT'] = chosen_data_root.as_posix()
    os.environ['LIBRI3MIX_ROOT_PATH'] = (chosen_data_root / 'LibriMix' / 'Libri3Mix').as_posix()
    print('CHIME2023_DATA_ROOT =', os.environ['CHIME2023_DATA_ROOT'])
    print('LIBRI3MIX_ROOT_PATH =', os.environ['LIBRI3MIX_ROOT_PATH'])

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
import os


def parse_pyver(ver_str):
    parts = ver_str.strip().split('.')
    return tuple(int(p) for p in parts[:2])


runner = [sys.executable]
pyver = sys.version.split()[0]
conda = shutil.which('conda')

if conda is not None:
    probe = subprocess.run(
        [conda, 'run', '-n', 'CHIME2023', 'python', '-c', 'import sys; print(sys.version.split()[0])'],
        capture_output=True,
        text=True,
    )
    if probe.returncode == 0:
        runner = [conda, 'run', '-n', 'CHIME2023', 'python']
        pyver = probe.stdout.strip()

strict_python_check = False
if parse_pyver(pyver) >= (3, 10):
    msg = (
        f'Aktiver Python ist {pyver}. fairseq in der George-Originalvariante ist oft nur mit 3.8/3.9 stabil. '
        'Ich versuche trotzdem den Run; falls fairseq crasht, brauchst du weiterhin eine 3.8/3.9-Umgebung.'
    )
    if strict_python_check:
        raise RuntimeError(msg)
    print('WARNUNG:', msg)

libri3mix_root = os.environ.get('LIBRI3MIX_ROOT_PATH', '').strip()
if not libri3mix_root:
    raise RuntimeError(
        'LIBRI3MIX_ROOT_PATH ist nicht gesetzt. Fuehre zuerst die Setup-Zelle aus, '
        'die den Datenpfad automatisch bestimmt.'
    )

expected_train_path = Path(libri3mix_root) / 'wav16k' / 'max' / 'train-360'
if not expected_train_path.exists():
    raise FileNotFoundError(
        f'Dataset-Pfad fehlt: {expected_train_path}. '
        'Setze LIBRI3MIX_ROOT_PATH auf einen gueltigen Libri3Mix-Root.'
    )
print(f'Dataset OK: {expected_train_path}')

# GPU-Auswahl: fuer physische GPU 1 hier "1" setzen.
selected_gpu = os.environ.get('CMGAN_GPU', '1').strip()
os.environ['CUDA_VISIBLE_DEVICES'] = selected_gpu
device_arg = 'cuda:0'
print(f'CUDA_VISIBLE_DEVICES={os.environ["CUDA_VISIBLE_DEVICES"]}, SpeechBrain device={device_arg}')

run_dir = (Path('runs') / hparams_file.stem).resolve()
run_dir.mkdir(parents=True, exist_ok=True)
overrides = [
    f'output_folder: {run_dir.as_posix()}/',
    'data_parallel_count: 1',
]
command = runner + ['train.py', str(hparams_file), '--device', device_arg] + overrides
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import os


def parse_pyver(ver_str):
    parts = ver_str.strip().split('.')
    return tuple(int(p) for p in parts[:2])


runner = [sys.executable]
pyver = sys.version.split()[0]
conda = shutil.which('conda')

if conda is not None:
    probe = subprocess.run(
        [conda, 'run', '-n', 'CHIME2023', 'python', '-c', 'import sys; print(sys.version.split()[0])'],
        capture_output=True,
        text=True,
    )
    if probe.returncode == 0:
        runner = [conda, 'run', '-n', 'CHIME2023', 'python']
        pyver = probe.stdout.strip()

strict_python_check = False
if parse_pyver(pyver) >= (3, 10):
    msg = (
        f'Aktiver Python ist {pyver}. fairseq in der George-Originalvariante ist oft nur mit 3.8/3.9 stabil. '
        'Ich versuche trotzdem den Batch-Run; falls fairseq crasht, brauchst du weiterhin eine 3.8/3.9-Umgebung.'
    )
    if strict_python_check:
        raise RuntimeError(msg)
    print('WARNUNG:', msg)

libri3mix_root = os.environ.get('LIBRI3MIX_ROOT_PATH', '').strip()
if not libri3mix_root:
    raise RuntimeError('LIBRI3MIX_ROOT_PATH ist nicht gesetzt. Fuehre zuerst die Setup-Zelle aus.')

expected_train_path = Path(libri3mix_root) / 'wav16k' / 'max' / 'train-360'
if not expected_train_path.exists():
    raise FileNotFoundError(f'Dataset-Pfad fehlt: {expected_train_path}')

# GPU-Auswahl fuer Batch-Run (physische GPU 1 -> "1").
selected_gpu = os.environ.get('CMGAN_GPU', '1').strip()
os.environ['CUDA_VISIBLE_DEVICES'] = selected_gpu
device_arg = 'cuda:0'
print(f'CUDA_VISIBLE_DEVICES={os.environ["CUDA_VISIBLE_DEVICES"]}, SpeechBrain device={device_arg}')

hparams_dir = Path('hparams')
if not hparams_dir.exists():
    raise FileNotFoundError('Ordner hparams nicht gefunden.')

# Primaer: train*.yaml; Fallback: hyperparams*.yaml
hparam_files = sorted(hparams_dir.glob('train*.yaml'))
if not hparam_files:
    hparam_files = sorted(hparams_dir.glob('hyperparams*.yaml'))

if not hparam_files:
    raise RuntimeError('Keine passenden Trainings-YAMLs gefunden (train*.yaml oder hyperparams*.yaml).')

print(f'Gefundene YAMLs: {len(hparam_files)}')
for idx, cfg in enumerate(hparam_files, start=1):
    run_dir = (Path('runs') / cfg.stem).resolve()
    run_dir.mkdir(parents=True, exist_ok=True)
    overrides = [
        f'output_folder: {run_dir.as_posix()}/',
        'data_parallel_count: 1',
    ]
    cmd = runner + ['train.py', str(cfg), '--device', device_arg] + overrides
    print(f'[{idx}/{len(hparam_files)}] ' + ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('Alle Trainingslaeufe abgeschlossen.')

## Batch-Run über alle Trainings-YAMLs

Diese Zelle läuft automatisch über alle Konfigurationsdateien in `hparams/`.
Priorität: zuerst `train*.yaml`, falls nichts gefunden wird `hyperparams*.yaml`.